# 02 — Calibration Analysis
Plot mm/pixel distribution across all 104 images. Verify ruler consistency.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.utils.io import load_config

cfg = load_config('../config.yaml')
cal_df = pd.read_csv(os.path.join('..', cfg['data']['calibration_csv']))
print(f'Loaded {len(cal_df)} calibration records')
cal_df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(cal_df['mm_per_pixel'], bins=20, edgecolor='black')
axes[0].set_xlabel('mm per pixel')
axes[0].set_title('mm/pixel distribution')

axes[1].hist(cal_df['distance_px'], bins=20, edgecolor='black', color='orange')
axes[1].set_xlabel('Ruler length (pixels)')
axes[1].set_title('Calibration ruler size distribution')

# Verify nearly-vertical ruler: Δx should be small relative to Δy
cal_df['delta_x'] = (cal_df['pt2_x'] - cal_df['pt1_x']).abs()
cal_df['delta_y'] = (cal_df['pt2_y'] - cal_df['pt1_y']).abs()
axes[2].scatter(cal_df['delta_x'], cal_df['delta_y'], alpha=0.7)
axes[2].set_xlabel('|Δx| pixels')
axes[2].set_ylabel('|Δy| pixels')
axes[2].set_title('Ruler orientation (should be mostly vertical)')

plt.tight_layout()
plt.show()

print(f"mm/pixel: mean={cal_df['mm_per_pixel'].mean():.4f}, std={cal_df['mm_per_pixel'].std():.4f}")
print(f"Ruler Δx: mean={cal_df['delta_x'].mean():.1f}px, max={cal_df['delta_x'].max():.1f}px")
print(f"Ruler Δy: mean={cal_df['delta_y'].mean():.1f}px, min={cal_df['delta_y'].min():.1f}px")

In [ ]:
# Check for any anomalous calibration values
mean_mpp = cal_df['mm_per_pixel'].mean()
std_mpp = cal_df['mm_per_pixel'].std()
outliers = cal_df[abs(cal_df['mm_per_pixel'] - mean_mpp) > 3 * std_mpp]
print(f'Outliers (>3σ from mean): {len(outliers)}')
if len(outliers):
    display(outliers[['image_id', 'mm_per_pixel', 'distance_px']])